## 1) Load Data

Load data, filter by removing unnecessary columns(data), and process the rest of it

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
pwd

In [ ]:
FILE_DIR = str(Path('path')) # Replace 'path' with the actual path to your directory

In [ ]:
df = pd.read_csv(FILE_DIR)

In [ ]:
bodyparts= df.iloc[0,1:]
coords = df.iloc[1,1:]

df_filtered = df.iloc[2:].reset_index(drop=True) # remove the first two rows which are not needed

flat_cols = ['frame'] + [
    f'{bp}_{c}' for bp, c in zip(bodyparts, coords)
]

df_filtered.columns = flat_cols
df_filtered = df_filtered.astype({col: float for col in df_filtered.columns if col != 'frame'})
df_filtered.drop(list(df_filtered.filter(regex='likelihood')), axis=1, inplace=True)

In [ ]:

df_filtered['TCTC_vector_x'] = df_filtered['TC_Middle_Leg_x'] - df_filtered['TC_Front_Leg_x'] 
df_filtered['TCTC_vector_y'] = df_filtered['TC_Middle_Leg_y'] - df_filtered['TC_Front_Leg_y']

df_filtered['TCFT_vector_x'] = df_filtered['TC_Middle_Leg_x'] - df_filtered['FT_Middle_Leg_x']
df_filtered['TCFT_vector_y'] = df_filtered['TC_Middle_Leg_y'] - df_filtered['FT_Middle_Leg_y']

# Dot product and magnitudes of the two vectors to get the cosine of the angle between them
df_filtered['dot_product'] = df_filtered['TCTC_vector_x'] * df_filtered['TCFT_vector_x'] + df_filtered['TCTC_vector_y'] * df_filtered['TCFT_vector_y']

df_filtered['TCTC_magnitude'] = (df_filtered['TCTC_vector_x']**2 + df_filtered['TCTC_vector_y']**2)**0.5
df_filtered['TCFT_magnitude'] = (df_filtered['TCFT_vector_x']**2 + df_filtered['TCFT_vector_y']**2)**0.5


df_filtered['cosine_angle'] = (df_filtered['dot_product'] / (df_filtered['TCTC_magnitude'] * df_filtered['TCFT_magnitude'])).clip(-1.0, 1.0)
df_filtered['angle'] = np.arccos(df_filtered['cosine_angle']) * 180 / np.pi

In [ ]:
df_filtered['angle'].plot(title='Angle between TCTC and TCFT vectors over time', xlabel='Frame', ylabel='Angle (degrees)')

In [ ]:
plt.figure(figsize=(10, 8))

# Plot the angles to see how they change over time
for idx, row in df_filtered.iterrows():
    x_coords = [row['TC_Front_Leg_x'], row['TC_Middle_Leg_x'], row['FT_Middle_Leg_x']]
    y_coords = [row['TC_Front_Leg_y'], row['TC_Middle_Leg_y'], row['FT_Middle_Leg_y']]
    
    plt.plot(x_coords, y_coords, color='blue', alpha=0.15)


# Highlight a middle frame in red
mid_row = df_filtered.iloc[len(df_filtered)//2]
plt.plot([mid_row['TC_Front_Leg_x'], mid_row['TC_Middle_Leg_x'], mid_row['FT_Middle_Leg_x']], 
         [mid_row['TC_Front_Leg_y'], mid_row['TC_Middle_Leg_y'], mid_row['FT_Middle_Leg_y']], 
         color='red', linewidth=2, label=f'Frame {len(df_filtered)//2}')

# Highlight the frame with the maximum angle in orange
max_angle = df_filtered.iloc[df_filtered['angle'].idxmax()]
plt.plot([max_angle['TC_Front_Leg_x'], max_angle['TC_Middle_Leg_x'], max_angle['FT_Middle_Leg_x']], 
         [max_angle['TC_Front_Leg_y'], max_angle['TC_Middle_Leg_y'], max_angle['FT_Middle_Leg_y']], 
         color='orange', linewidth=2, label=f'Max Angle Frame {df_filtered["angle"].idxmax()}')


plt.title('Connected Leg Segments Showing Range of Motion (All Frames)')
plt.xlabel('X Coordinate (Pixels)')
plt.ylabel('Y Coordinate (Pixels)')
plt.legend()
plt.grid(True)

# Note: Image data usually has the Y-axis flipped (0 at the top).
plt.gca().invert_yaxis() 

plt.tight_layout()
plt.show()